In [1]:
from pathlib import Path
import subprocess
import time
import requests

PROJECT_ROOT = Path(r"X:\dev\projects\study\ATLAS")
LLAMA_DIR = PROJECT_ROOT / "llama"
MODEL_PATH = PROJECT_ROOT / "models" / "gemma3" / "gemma-3-4b-it-Q4_K_M.gguf"
LOG_PATH = PROJECT_ROOT / "rag" / "llama_server.log"

HOST = "127.0.0.1"
PORT = 8080

In [2]:
SERVER_CMD = [
    str(LLAMA_DIR / "llama-server.exe"),
    "-m", str(MODEL_PATH),
    "-ngl", "999",
    "-c", "8192",
    "--host", HOST,
    "--port", str(PORT),
]

log_f = open(LOG_PATH, "w", encoding="utf-8")
llama_proc = subprocess.Popen(
    SERVER_CMD,
    cwd=str(LLAMA_DIR),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    creationflags=subprocess.CREATE_NEW_PROCESS_GROUP,
)
print(f"llama-server PID={llama_proc.pid}")

for _ in range(30):
    try:
        requests.get(f"http://{HOST}:{PORT}/health", timeout=2)
        print("server ready")
        break
    except Exception:
        time.sleep(2)
else:
    print("server did not start in time, check the log")

llama-server PID=26024
server ready


In [3]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from rag.search import close_index, load_index, rag_answer

idx = load_index(PROJECT_ROOT / "data" / "index")

x:\dev\projects\study\ATLAS\.venv\Lib\site-packages\transformers\utils\hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-embeddings-v3:
- custom_st.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/jinaai/xlm-roberta-flash-implementation:
- configuration_xlm_roberta.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/jinaai/xlm-roberta-flash-implementation:
- modeling_lora.py
. Make sure to double-check they do not contain any added malici

In [4]:
question = "what do the 3 dots mean in math"
answer, sources = rag_answer(question, idx)

print(f"Q: {question}")
print(f"A: {answer}")
print()
for s in sources:
    print(f"  [{s['title']}] chunk {s['chunk_idx']}")

full prompt: Answer the question using ONLY the provided context. Be concise.

Context:
[1] Anno Domini
rder. The abbreviation is also widely used after the number of a century or millennium, as in "fourth century AD" or "second millennium AD" (although conservative usage formerly rejected such expressions). Because BC is the English abbreviation for Before Christ, it is sometimes incorrectly concluded that AD means After Death, i.e., after the death of Jesus. However, this would mean that the approximate 33 years commonly associated with the life of Jesus would neither be included in the BC nor t

[2] National identification number
n for females. The final character, C, is a checksum value over the first 17 digits. To calculate the checksum, each digit in order is multiplied by a weight in the ordered set [7 9 10 5 8 4 2 1 6 3 7 9 10 5 8 4 2] and summed together. The sum modulus 11 is used as an index into the ordered set [1 0 X 9 8 7 6 5 4 3 2], with the first index being zero. The i

In [5]:
question = "who wrote the song photograph by ringo starr"
answer, sources = rag_answer(question, idx)

print(f"Q: {question}")
print(f"A: {answer}")
print()
for s in sources:
    print(f"  [{s['title']}] chunk {s['chunk_idx']}")

full prompt: Answer the question using ONLY the provided context. Be concise.

Context:
[1] The Book of Love (song)
ne and Charles Patrick.

Lead singer Charles Patrick heard a Pepsodent toothpaste commercial with the line "wonder where the yellow went". From there he got the idea for the line, "I wonder, wonder, wonder who, who wrote the book of love", working it up into a song with Davis and Malone. The "boom" part of the song was a result of a kid kicking a ball against the garage while they were rehearsing. It sounded good, so they added it to the song. 

In September 1957, the Monotones recorded "The Boo

[2] Piano Man (song)
| format = 45 rpm single
| recorded = September 1973
| studio =
| venue =
| genre =
* Soft rock
* folk rock
| length = (Album version)4:30 (Single version)
| label = Columbia
| writer = Billy Joel
| producer = Michael Stewart
| prev_title = She's Got a Way
| prev_year = 1972
| next_title = Worse Comes to Worst
| next_year = 1974
| misc = 
}}
}}

"Piano Man" i

In [6]:
import gc

if llama_proc.poll() is None:
    llama_proc.terminate()
    try:
        llama_proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        llama_proc.kill()
        llama_proc.wait()

log_f.close()

close_index(idx)
del idx
gc.collect()

print(f"server stopped (exit code {llama_proc.returncode})")

server stopped (exit code 1)
